# QRGuard Semantic Training — complete Google Colab run

Acquires and freezes URL corpora, removes conflicts, creates a registrable-domain-disjoint holdout, trains the serving-compatible calibrated character 3–5 gram model, runs behavioural gates, and exports every Semantic performance artifact. A GPU is not required for this sparse linear model.

## Phase 0 — Reproducible workspace and Drive mount

In [ ]:
# Mount Drive and unpack the exact source bundle.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

BUNDLE_ZIP = Path('/content/drive/MyDrive/QRGuard_ML_Colab.zip')
WORK = Path('/content/qrguard_ml')
if not BUNDLE_ZIP.is_file():
    raise FileNotFoundError(
        f'Upload QRGuard_ML_Colab.zip to {BUNDLE_ZIP} before Run all.'
    )
print('Bundle SHA-256:', hashlib.sha256(BUNDLE_ZIP.read_bytes()).hexdigest().upper())
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as archive:
    archive.extractall(WORK)
REPO = WORK / 'QRGuard_ML_Colab' / 'QRGuard'
assert (REPO / 'ml_training/requirements.txt').is_file(), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'backend'))
print('Training source:', REPO)


## Phase 1 — Environment and dependency audit

In [ ]:
# Install QRGuard dependencies without replacing Colab's matched
# CUDA-enabled torch/torchvision wheels. Re-resolving only one side of that pair
# can make torchvision fail during import even though `import torch` still works.
def torch_runtime_smoke_test(stage):
    try:
        import torch, torchvision
        from torchvision import models, transforms
        probe = models.resnet18(weights=None)
        assert probe.fc.in_features == 512 and transforms.ToTensor is not None
    except Exception as exc:
        raise RuntimeError(
            f'Colab torch/torchvision is inconsistent {stage}: {exc}\n'
            'Choose Runtime > Disconnect and delete runtime, reconnect with a T4 GPU, '
            'then run this updated notebook from Phase 0.'
        ) from exc
    print(
        f'PyTorch runtime {stage}: torch={torch.__version__}, '
        f'torchvision={torchvision.__version__}, CUDA={torch.cuda.is_available()}'
    )

torch_runtime_smoke_test('before dependency install')
requirements = REPO / 'ml_training/requirements.txt'
colab_requirements = Path('/tmp/qrguard_colab_requirements.txt')
protected = {'torch', 'torchvision'}
lines = []
for line in requirements.read_text(encoding='utf-8').splitlines():
    package = line.split(';', 1)[0].split('[', 1)[0]
    package = package.split('=', 1)[0].split('<', 1)[0].split('>', 1)[0].strip().lower()
    if package not in protected:
        lines.append(line)
colab_requirements.write_text('\n'.join(lines) + '\n', encoding='utf-8')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(colab_requirements),
])
torch_runtime_smoke_test('after dependency install')

def run_module(module, *arguments, check=True):
    # Colab can suppress output inherited by subprocess.run. Merge and stream
    # both channels explicitly so the real child traceback is never replaced by
    # an unhelpful outer CalledProcessError.
    command = [sys.executable, '-u', '-m', module, *map(str, arguments)]
    print('>', ' '.join(command), flush=True)
    environment = os.environ.copy()
    environment['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        command,
        cwd=REPO,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        output.append(line)
    returncode = process.wait()
    result = subprocess.CompletedProcess(
        command, returncode, stdout=''.join(output), stderr=None
    )
    if check and returncode:
        raise RuntimeError(
            f'{module} failed with return code {returncode}; '
            'the complete child output is printed immediately above.'
        )
    return result

run_module('ml_training.scripts.audit_environment')


## Phase 2 — Acquire, standardise and freeze Semantic sources

PhiUSIIL is fetched from UCI, Malicious URLs through KaggleHub, and a dated Tranco list through the Tranco client. Prepared data is cached in Drive.

In [ ]:
SEMANTIC_DRIVE = Path('/content/drive/MyDrive/QRGuard_ML_Data/semantic')
LOCAL_DATA = REPO / 'data/method1'
if (SEMANTIC_DRIVE / 'provenance.json').is_file():
    shutil.copytree(SEMANTIC_DRIVE, LOCAL_DATA, dirs_exist_ok=True)
    print('Restored prepared Semantic data from Drive')
else:
    run_module('ml_training.semantic.src.prepare_colab_data')
    SEMANTIC_DRIVE.mkdir(parents=True, exist_ok=True)
    shutil.copytree(LOCAL_DATA, SEMANTIC_DRIVE, dirs_exist_ok=True)
provenance = json.loads((LOCAL_DATA / 'provenance.json').read_text())
display(provenance)


## Phase 3 — Provenance, labels and domain-disjoint holdout audit

In [ ]:
import pandas as pd
heldout = pd.read_parquet(LOCAL_DATA / 'heldout_test.parquet')
print('Held-out rows:', len(heldout))
print('Held-out registrable domains:', heldout.domain.nunique())
display(heldout.groupby(['source', 'label']).size().rename('rows').reset_index())
assert heldout.domain.notna().all()
assert set(heldout.label.unique()).issubset({0, 1})


## Phase 4 — Clean conflicts, grouped split, train and calibrate

The runtime-compatible model uses the same URL enrichment and hashing contract as the backend. Validation fits calibration; test data is not used to train parameters.

In [ ]:
training = run_module('ml_training.semantic.src.train_local', check=False)
print('Training return code:', training.returncode)


## Phase 5 — Display every Semantic performance output

In [ ]:
from IPython.display import Image as DisplayImage, Markdown, display
PERF = REPO / 'ml_training/semantic/performance/semantic-2026.02'
metrics = json.loads((PERF / 'metrics.json').read_text())
display(Markdown((PERF / 'SEMANTIC_PERFORMANCE.md').read_text()))
display(metrics)
for name in ('training_curves.png', 'confusion_matrix.png', 'roc_pr_curves.png',
             'calibration_curve.png'):
    display(Markdown(f'### {name}'))
    display(DisplayImage(filename=str(PERF / name)))
for name in ('metrics.csv', 'dataset_composition.csv', 'threshold_analysis.csv',
             'per_source_results.csv', 'hard_benign_results.csv',
             'behavioural_acceptance.csv'):
    print('\n', name)
    display(pd.read_csv(PERF / name).head(50))


## Phase 6 — Validate serving parity, gates and report completeness

In [ ]:
validation = run_module(
    'ml_training.scripts.validate_performance_bundle',
    '--branch', 'semantic',
    check=False,
)
if validation.returncode:
    raise RuntimeError('Semantic report bundle is incomplete; inspect the phase above.')
summary = json.loads((REPO / 'ml_training/PERFORMANCE_VALIDATION.json').read_text())
display(summary)


## Phase 7 — Save model and all performance outputs to Drive

In [ ]:
destination = Path('/content/drive/MyDrive/QRGuard_ML_Results/semantic-2026.02')
destination.mkdir(parents=True, exist_ok=True)
shutil.copytree(PERF, destination / 'performance', dirs_exist_ok=True)
shutil.copytree(
    REPO / 'ml_training/semantic/runs/semantic-2026.02/artifacts',
    destination / 'artifacts',
    dirs_exist_ok=True,
)
shutil.copy2(REPO / 'ml_training/PERFORMANCE_VALIDATION.json', destination)
archive = shutil.make_archive(str(destination), 'zip', root_dir=destination)
print('Saved:', destination)
print('Archive:', archive)
